# 💾 Analyse des Données depuis TimescaleDB

Ce notebook analyse les données crypto stockées dans la base de données TimescaleDB.

## Objectifs
1. Se connecter à la base de données
2. Explorer les données avec SQL
3. Analyser les statistiques avancées
4. Calculer les indicateurs techniques
5. Visualiser les données avec Plotly
6. Comparer les performances des cryptos

In [1]:
# Imports
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import psycopg2
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# Ajouter le dossier racine au path
sys.path.insert(0, str(Path.cwd().parent))

from src.utils.config import get_config

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Imports réussis!")
print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Imports réussis!
📅 Date: 2025-11-27 09:17:25


## 1. 🔌 Connexion à la Base de Données

In [3]:
# Charger la configuration
config = get_config()

# Connexion à PostgreSQL
def get_connection():
    return psycopg2.connect(
        host=config.database.host,
        port=config.database.port,
        database=config.database.name,
        user=config.database.user,
        password=config.database.password
    )

# Fonction helper pour exécuter des requêtes
def query_to_df(query, params=None):
    """Exécute une requête SQL et retourne un DataFrame."""
    conn = get_connection()
    try:
        df = pd.read_sql_query(query, conn, params=params)
        return df
    finally:
        conn.close()

# Test de connexion
try:
    conn = get_connection()
    print("✅ Connexion à TimescaleDB réussie!")
    print(f"📊 Base: {config.database.name}")
    print(f"🏠 Host: {config.database.host}")
    conn.close()
except Exception as e:
    print(f"❌ Erreur de connexion: {e}")

✅ Connexion à TimescaleDB réussie!
📊 Base: crypto_prediction
🏠 Host: localhost


## 2. 📊 Vue d'Ensemble des Données

In [4]:
# Statistiques générales
stats_query = """
SELECT 
    COUNT(DISTINCT symbol) as nb_cryptos,
    COUNT(*) as total_candles,
    MIN(time) as first_date,
    MAX(time) as last_date,
    pg_size_pretty(pg_total_relation_size('ohlcv_data')) as table_size
FROM ohlcv_data;
"""

stats = query_to_df(stats_query)
print("="*70)
print("📈 STATISTIQUES GÉNÉRALES")
print("="*70)
display(stats)

# Durée de la période
duration = stats['last_date'].iloc[0] - stats['first_date'].iloc[0]
print(f"\n⏱️  Période couverte: {duration.days} jours")

📈 STATISTIQUES GÉNÉRALES


,nb_cryptos,total_candles,first_date,last_date,table_size
0,11,8759,2025-08-23 03:00:00+00:00,2025-11-23 06:00:00+00:00,40 kB



⏱️  Période couverte: 92 jours


In [5]:
# Données par crypto
crypto_stats_query = """
SELECT 
    symbol,
    COUNT(*) as nb_candles,
    MIN(time) as first_date,
    MAX(time) as last_date,
    ROUND(AVG(close)::numeric, 2) as avg_price,
    ROUND(MIN(low)::numeric, 2) as min_price,
    ROUND(MAX(high)::numeric, 2) as max_price,
    ROUND(SUM(volume)::numeric, 2) as total_volume
FROM ohlcv_data
GROUP BY symbol
ORDER BY total_volume DESC;
"""

crypto_stats = query_to_df(crypto_stats_query)
print("\n📊 STATISTIQUES PAR CRYPTO")
print("="*70)
display(crypto_stats)


📊 STATISTIQUES PAR CRYPTO


,symbol,nb_candles,first_date,last_date,avg_price,min_price,max_price,total_volume
0,ADA/USDT,720,2025-10-24 07:00:00+00:00,2025-11-23 06:00:00+00:00,0.56,0.39,0.69,4476415414.40
1,USDT/TRY,772,2025-10-22 01:00:00+00:00,2025-11-23 04:00:00+00:00,42.20,41.82,42.60,1748536344.00
2,UNI/USDT,720,2025-10-24 05:00:00+00:00,2025-11-23 04:00:00+00:00,6.55,4.74,10.30,331461202.55
3,DOT/USDT,720,2025-10-24 05:00:00+00:00,2025-11-23 04:00:00+00:00,2.89,2.25,3.53,283105721.11
4,LINK/USDT,720,2025-10-24 05:00:00+00:00,2025-11-23 04:00:00+00:00,15.58,11.61,19.06,148441112.12
5,SOL/USDT,720,2025-10-24 07:00:00+00:00,2025-11-23 06:00:00+00:00,163.41,121.66,205.33,128399578.95
6,AVAX/USDT,720,2025-10-24 05:00:00+00:00,2025-11-23 04:00:00+00:00,17.05,12.57,21.10,101781180.52
7,USDT/ARS,720,2025-10-22 01:00:00+00:00,2025-11-21 00:00:00+00:00,1501.30,1390.00,1607.30,64711566.00
8,ETH/USDT,1093,2025-08-23 03:00:00+00:00,2025-11-23 06:00:00+00:00,3760.99,2623.57,4956.78,62043374.72
9,BNB/USDT,720,2025-10-24 07:00:00+00:00,2025-11-23 06:00:00+00:00,993.98,790.79,1182.60,10933481.76


## 3. 🔍 Requêtes SQL Avancées avec TimescaleDB

In [6]:
# Utiliser time_bucket pour agréger par jour
daily_query = """
SELECT 
    time_bucket('1 day', time) AS day,
    symbol,
    FIRST(open, time) as day_open,
    MAX(high) as day_high,
    MIN(low) as day_low,
    LAST(close, time) as day_close,
    SUM(volume) as day_volume
FROM ohlcv_data
GROUP BY day, symbol
ORDER BY day DESC, symbol
LIMIT 50;
"""

daily_data = query_to_df(daily_query)
print("📅 DONNÉES QUOTIDIENNES (time_bucket)")
print("="*70)
display(daily_data.head(10))

📅 DONNÉES QUOTIDIENNES (time_bucket)


,day,symbol,day_open,day_high,day_low,day_close,day_volume
0,2025-11-23 00:00:00+00:00,ADA/USDT,0.40,0.42,0.40,0.41,26980219.30
1,2025-11-23 00:00:00+00:00,AVAX/USDT,13.24,13.44,13.23,13.33,436463.08
2,2025-11-23 00:00:00+00:00,BNB/USDT,835.77,854.49,835.00,841.50,62832.52
3,2025-11-23 00:00:00+00:00,BTC/USDT,84971.74,86860.00,84950.97,85747.44,5539.03
4,2025-11-23 00:00:00+00:00,DOT/USDT,2.31,2.37,2.31,2.35,1104707.51
5,2025-11-23 00:00:00+00:00,ETH/USDT,2773.00,2852.17,2771.93,2796.99,129282.20
6,2025-11-23 00:00:00+00:00,LINK/USDT,12.15,12.60,12.14,12.48,824583.51
7,2025-11-23 00:00:00+00:00,SOL/USDT,128.04,131.90,127.88,128.81,769075.24
8,2025-11-23 00:00:00+00:00,UNI/USDT,6.11,6.36,6.10,6.28,863595.51
9,2025-11-23 00:00:00+00:00,USDT/TRY,42.58,42.59,42.57,42.58,2223719.00


In [7]:
# Performance des dernières 24h
performance_24h_query = """
WITH latest AS (
    SELECT DISTINCT ON (symbol)
        symbol,
        close as current_price,
        time as current_time
    FROM ohlcv_data
    ORDER BY symbol, time DESC
),
day_ago AS (
    SELECT DISTINCT ON (symbol)
        symbol,
        close as price_24h_ago
    FROM ohlcv_data
    WHERE time <= NOW() - INTERVAL '24 hours'
    ORDER BY symbol, time DESC
)
SELECT 
    l.symbol,
    ROUND(l.current_price::numeric, 2) as current_price,
    ROUND(d.price_24h_ago::numeric, 2) as price_24h_ago,
    ROUND(((l.current_price - d.price_24h_ago) / d.price_24h_ago * 100)::numeric, 2) as change_24h_pct
FROM latest l
JOIN day_ago d ON l.symbol = d.symbol
ORDER BY change_24h_pct DESC;
"""

performance = query_to_df(performance_24h_query)
print("\n📈 PERFORMANCE 24H")
print("="*70)
display(performance)


📈 PERFORMANCE 24H


,symbol,current_price,price_24h_ago,change_24h_pct
0,ADA/USDT,0.41,0.41,0.00
1,AVAX/USDT,13.33,13.33,0.00
2,BNB/USDT,841.50,841.50,0.00
3,BTC/USDT,85747.44,85747.44,0.00
4,DOT/USDT,2.35,2.35,0.00
5,ETH/USDT,2796.99,2796.99,0.00
6,LINK/USDT,12.48,12.48,0.00
7,SOL/USDT,128.81,128.81,0.00
8,UNI/USDT,6.28,6.28,0.00
9,USDT/ARS,1492.40,1492.40,0.00


## 4. 📥 Chargement des Données pour Analyse

In [8]:
# Charger toutes les données
def load_crypto_data(symbol=None, limit=None):
    """Charge les données OHLCV pour une ou toutes les cryptos."""
    query = "SELECT * FROM ohlcv_data"
    
    if symbol:
        query += f" WHERE symbol = '{symbol}'"
    
    query += " ORDER BY time"
    
    if limit:
        query += f" LIMIT {limit}"
    
    df = query_to_df(query)
    df['time'] = pd.to_datetime(df['time'])
    return df

# Charger les données
all_data = load_crypto_data()
print(f"✅ {len(all_data)} bougies chargées")
print(f"📊 Cryptos: {all_data['symbol'].unique()}")
print(f"📅 Période: {all_data['time'].min()} à {all_data['time'].max()}")

display(all_data.head())

✅ 8759 bougies chargées
📊 Cryptos: ['BTC/USDT' 'ETH/USDT' 'USDT/ARS' 'USDT/TRY' 'LINK/USDT' 'DOT/USDT'
 'AVAX/USDT' 'UNI/USDT' 'SOL/USDT' 'BNB/USDT' 'ADA/USDT']
📅 Période: 2025-08-23 03:00:00+00:00 à 2025-11-23 06:00:00+00:00


,time,symbol,open,high,low,close,volume,quote_volume,trades_count,timeframe,created_at
0,2025-08-23 03:00:00+00:00,BTC/USDT,115568.77,116034.53,115551.20,115799.47,2258.18,0.00,0,1h,2025-11-27 08:06:07.054803+00:00
1,2025-08-23 03:00:00+00:00,ETH/USDT,4694.63,4765.54,4694.33,4713.88,102546.79,0.00,0,1h,2025-11-27 08:06:05.072388+00:00
2,2025-08-23 07:00:00+00:00,BTC/USDT,115799.46,115861.80,115223.43,115310.13,1313.89,0.00,0,1h,2025-11-27 08:06:07.054803+00:00
3,2025-08-23 07:00:00+00:00,ETH/USDT,4713.89,4746.09,4690.72,4704.93,55489.62,0.00,0,1h,2025-11-27 08:06:05.072388+00:00
4,2025-08-23 11:00:00+00:00,BTC/USDT,115310.13,115486.97,114560.00,114812.87,3027.56,0.00,0,1h,2025-11-27 08:06:07.054803+00:00


## 5. 📊 Visualisations Interactives

In [9]:
# Graphique des prix
fig = go.Figure()

for symbol in all_data['symbol'].unique():
    df_symbol = all_data[all_data['symbol'] == symbol]
    
    fig.add_trace(go.Scatter(
        x=df_symbol['time'],
        y=df_symbol['close'],
        mode='lines',
        name=symbol,
        hovertemplate='<b>%{fullData.name}</b><br>' +
                      'Date: %{x}<br>' +
                      'Prix: $%{y:.2f}<br>' +
                      '<extra></extra>'
    ))

fig.update_layout(
    title='📈 Évolution des Prix (depuis TimescaleDB)',
    xaxis_title='Date',
    yaxis_title='Prix (USD)',
    height=600,
    hovermode='x unified',
    template='plotly_dark'
)

fig.show()

In [10]:
# Graphique en chandelier pour chaque crypto
for symbol in all_data['symbol'].unique():
    df_symbol = all_data[all_data['symbol'] == symbol].copy()
    
    fig = go.Figure(data=[go.Candlestick(
        x=df_symbol['time'],
        open=df_symbol['open'],
        high=df_symbol['high'],
        low=df_symbol['low'],
        close=df_symbol['close'],
        name=symbol
    )])
    
    fig.update_layout(
        title=f'🕯️ Chandelier - {symbol}',
        xaxis_title='Date',
        yaxis_title='Prix (USD)',
        height=500,
        template='plotly_dark',
        xaxis_rangeslider_visible=False
    )
    
    fig.show()

## 6. 📊 Analyse des Volumes

In [11]:
# Volume analysis avec SQL
volume_query = """
SELECT 
    time,
    symbol,
    volume,
    AVG(volume) OVER (
        PARTITION BY symbol 
        ORDER BY time 
        ROWS BETWEEN 23 PRECEDING AND CURRENT ROW
    ) as volume_sma_24
FROM ohlcv_data
ORDER BY time;
"""

volume_data = query_to_df(volume_query)

# Graphique des volumes
for symbol in volume_data['symbol'].unique():
    df_vol = volume_data[volume_data['symbol'] == symbol]
    
    fig = make_subplots(
        rows=2, cols=1,
        row_heights=[0.7, 0.3],
        subplot_titles=(f'{symbol} - Prix', 'Volume'),
        vertical_spacing=0.1
    )
    
    # Prix
    df_price = all_data[all_data['symbol'] == symbol]
    fig.add_trace(
        go.Scatter(x=df_price['time'], y=df_price['close'], name='Prix'),
        row=1, col=1
    )
    
    # Volume
    fig.add_trace(
        go.Bar(x=df_vol['time'], y=df_vol['volume'], name='Volume', marker_color='lightblue'),
        row=2, col=1
    )
    
    # Volume SMA
    fig.add_trace(
        go.Scatter(x=df_vol['time'], y=df_vol['volume_sma_24'], name='Volume SMA 24', line=dict(color='red')),
        row=2, col=1
    )
    
    fig.update_layout(height=600, template='plotly_dark', showlegend=True)
    fig.show()

## 7. 📈 Calcul des Indicateurs Techniques avec SQL

In [12]:
# Calculer les moyennes mobiles avec SQL (Window Functions)
indicators_query = """
SELECT 
    time,
    symbol,
    close,
    AVG(close) OVER (PARTITION BY symbol ORDER BY time ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) as sma_7,
    AVG(close) OVER (PARTITION BY symbol ORDER BY time ROWS BETWEEN 24 PRECEDING AND CURRENT ROW) as sma_25,
    AVG(close) OVER (PARTITION BY symbol ORDER BY time ROWS BETWEEN 98 PRECEDING AND CURRENT ROW) as sma_99
FROM ohlcv_data
ORDER BY symbol, time;
"""

indicators = query_to_df(indicators_query)
print("✅ Indicateurs calculés")
display(indicators.tail(10))

✅ Indicateurs calculés


,time,symbol,close,sma_7,sma_25,sma_99
8749,2025-11-22 19:00:00+00:00,USDT/TRY,42.59,42.59,42.58,42.43
8750,2025-11-22 20:00:00+00:00,USDT/TRY,42.59,42.59,42.58,42.44
8751,2025-11-22 21:00:00+00:00,USDT/TRY,42.59,42.59,42.59,42.44
8752,2025-11-22 22:00:00+00:00,USDT/TRY,42.58,42.59,42.59,42.44
8753,2025-11-22 23:00:00+00:00,USDT/TRY,42.59,42.59,42.59,42.44
8754,2025-11-23 00:00:00+00:00,USDT/TRY,42.59,42.59,42.59,42.45
8755,2025-11-23 01:00:00+00:00,USDT/TRY,42.59,42.59,42.59,42.45
8756,2025-11-23 02:00:00+00:00,USDT/TRY,42.58,42.59,42.59,42.45
8757,2025-11-23 03:00:00+00:00,USDT/TRY,42.58,42.59,42.59,42.45
8758,2025-11-23 04:00:00+00:00,USDT/TRY,42.58,42.58,42.59,42.45


In [13]:
# Visualiser les moyennes mobiles
for symbol in indicators['symbol'].unique():
    df_ind = indicators[indicators['symbol'] == symbol].copy()
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(x=df_ind['time'], y=df_ind['close'], name='Prix', line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df_ind['time'], y=df_ind['sma_7'], name='SMA 7', line=dict(dash='dash')))
    fig.add_trace(go.Scatter(x=df_ind['time'], y=df_ind['sma_25'], name='SMA 25', line=dict(dash='dash')))
    fig.add_trace(go.Scatter(x=df_ind['time'], y=df_ind['sma_99'], name='SMA 99', line=dict(dash='dot')))
    
    fig.update_layout(
        title=f'📊 {symbol} - Prix et Moyennes Mobiles',
        xaxis_title='Date',
        yaxis_title='Prix (USD)',
        height=600,
        template='plotly_dark',
        hovermode='x unified'
    )
    
    fig.show()

## 8. 💹 Analyse des Rendements

In [14]:
# Calculer les rendements avec SQL
returns_query = """
SELECT 
    time,
    symbol,
    close,
    LAG(close) OVER (PARTITION BY symbol ORDER BY time) as prev_close,
    (close - LAG(close) OVER (PARTITION BY symbol ORDER BY time)) / 
        LAG(close) OVER (PARTITION BY symbol ORDER BY time) * 100 as returns_pct
FROM ohlcv_data
ORDER BY symbol, time;
"""

returns_data = query_to_df(returns_query)
returns_data = returns_data.dropna()

print("📈 STATISTIQUES DES RENDEMENTS")
print("="*70)
stats_returns = returns_data.groupby('symbol')['returns_pct'].agg([
    ('Moyenne', 'mean'),
    ('Médiane', 'median'),
    ('Écart-type', 'std'),
    ('Min', 'min'),
    ('Max', 'max')
]).round(4)

display(stats_returns)

📈 STATISTIQUES DES RENDEMENTS


,Moyenne,Médiane,Écart-type,Min,Max
symbol,,,,,
ADA/USDT,-0.06,-0.04,0.94,-4.79,4.26
AVAX/USDT,-0.05,0.00,0.97,-6.51,3.55
BNB/USDT,-0.04,0.01,0.69,-4.15,2.89
BTC/USDT,-0.02,-0.01,0.61,-3.33,4.46
DOT/USDT,-0.03,0.00,1.12,-6.92,7.50
ETH/USDT,-0.04,-0.03,1.19,-7.89,5.61
LINK/USDT,-0.04,0.00,1.00,-6.25,3.18
SOL/USDT,-0.05,0.00,0.92,-5.47,2.86
UNI/USDT,0.01,-0.06,1.52,-5.21,12.74


In [15]:
# Distribution des rendements
for symbol in returns_data['symbol'].unique():
    df_ret = returns_data[returns_data['symbol'] == symbol]
    
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=df_ret['returns_pct'],
        nbinsx=50,
        name=symbol
    ))
    
    fig.update_layout(
        title=f'📊 {symbol} - Distribution des Rendements',
        xaxis_title='Rendement (%)',
        yaxis_title='Fréquence',
        height=400,
        template='plotly_dark'
    )
    
    fig.show()

## 9. 🔗 Analyse des Corrélations

In [17]:
# Créer une matrice de corrélation
if len(all_data['symbol'].unique()) > 1:
    # Pivot pour avoir les prix en colonnes
    price_matrix = all_data.pivot(index='time', columns='symbol', values='close')
    
    # Calculer les rendements
    returns_matrix = price_matrix.pct_change().dropna()
    
    # Corrélation
    corr = returns_matrix.corr()
    
    # Heatmap
    fig = go.Figure(data=go.Heatmap(
        z=corr.values,
        x=corr.columns,
        y=corr.index,
        colorscale='RdBu',
        zmid=0,
        text=corr.values,
        texttemplate='%{text:.2f}',
        textfont={"size": 12}
    ))
    
    fig.update_layout(
        title='🔗 Matrice de Corrélation des Rendements',
        height=500,
        template='plotly_dark'
    )
    
    fig.show()
else:
    print("⚠️  Besoin d'au moins 2 cryptos pour calculer les corrélations")

## 10. 📝 Résumé et Export

In [18]:
# Résumé final
summary_query = """
SELECT 
    symbol,
    COUNT(*) as nb_candles,
    ROUND(FIRST(close, time)::numeric, 2) as first_price,
    ROUND(LAST(close, time)::numeric, 2) as last_price,
    ROUND(((LAST(close, time) - FIRST(close, time)) / FIRST(close, time) * 100)::numeric, 2) as total_change_pct,
    ROUND(MIN(low)::numeric, 2) as min_price,
    ROUND(MAX(high)::numeric, 2) as max_price,
    ROUND(AVG(volume)::numeric, 2) as avg_volume,
    ROUND(SUM(volume)::numeric, 2) as total_volume
FROM ohlcv_data
GROUP BY symbol
ORDER BY total_volume DESC;
"""

summary = query_to_df(summary_query)

print("="*70)
print("📊 RÉSUMÉ COMPLET")
print("="*70)
display(summary)

print("\n✅ Analyse terminée!")

📊 RÉSUMÉ COMPLET


,symbol,nb_candles,first_price,last_price,total_change_pct,min_price,max_price,avg_volume,total_volume
0,ADA/USDT,720,0.65,0.41,-37.35,0.39,0.69,6217243.63,4476415414.40
1,USDT/TRY,772,42.02,42.58,1.33,41.82,42.60,2264943.45,1748536344.00
2,UNI/USDT,720,6.40,6.28,-1.86,4.74,10.30,460362.78,331461202.55
3,DOT/USDT,720,3.06,2.35,-23.29,2.25,3.53,393202.39,283105721.11
4,LINK/USDT,720,17.70,12.48,-29.49,11.61,19.06,206168.21,148441112.12
5,SOL/USDT,720,192.97,128.81,-33.25,121.66,205.33,178332.75,128399578.95
6,AVAX/USDT,720,19.61,13.33,-32.02,12.57,21.10,141362.75,101781180.52
7,USDT/ARS,720,1590.00,1492.40,-6.14,1390.00,1607.30,89877.18,64711566.00
8,ETH/USDT,1093,4713.88,2796.99,-40.66,2623.57,4956.78,56764.30,62043374.72
9,BNB/USDT,720,1128.06,841.50,-25.40,790.79,1182.60,15185.39,10933481.76



✅ Analyse terminée!


In [19]:
# Sauvegarder le résumé
output_dir = Path('../output')
output_dir.mkdir(exist_ok=True)

summary.to_csv(output_dir / 'database_summary.csv', index=False)
print(f"💾 Résumé sauvegardé dans: {output_dir / 'database_summary.csv'}")

💾 Résumé sauvegardé dans: ../output/database_summary.csv


## 🎯 Prochaines Étapes

1. **Feature Engineering** - Calculer et stocker les indicateurs dans la table `features`
2. **Modélisation ML** - Entraîner les premiers modèles
3. **API** - Créer une API REST pour accéder aux données
4. **Dashboard** - Créer un dashboard Streamlit temps réel

---

**📝 Notes:**
- Toutes les analyses utilisent TimescaleDB
- Les window functions SQL sont optimisées
- Les données sont prêtes pour le ML